# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [3]:
context = [ {'role':'system', 'content':"""
You are a bot to assist in creating SQL commands.
All your answers should be valid SQLite.
Keep the SQL simple and efficient.
Only return the SQL query, no explanation needed.

The database is composed of the following tables:

CREATE TABLE employees (
    ID_Usr INTEGER PRIMARY KEY,  -- Unique ID for each employee
    name VARCHAR(50),            -- Full name of the employee
    department VARCHAR(50),      -- Department the employee belongs to
    hire_date DATE               -- Date the employee was hired
);

/* Sample rows:
ID_Usr | name            | department  | hire_date
1      | Alice Johnson   | Engineering | 2018-03-15
2      | Bob Smith       | Marketing   | 2020-07-01
3      | Clara Mendes    | Engineering | 2016-11-22
*/

CREATE TABLE salary (
    ID_Usr INTEGER,       -- References employees.ID_Usr
    year INTEGER,         -- Year of the salary record
    salary FLOAT,         -- Annual salary amount
    bonus FLOAT           -- Annual bonus amount
);

/* Sample rows:
ID_Usr | year | salary   | bonus
1      | 2023 | 72000.0  | 5000.0
2      | 2023 | 55000.0  | 2000.0
3      | 2023 | 89000.0  | 8000.0
*/

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,       -- Unique ID for the study record
    ID_Usr INTEGER,               -- References employees.ID_Usr
    educational_level INTEGER,    -- 1=Bachelor, 2=Master, 3=PhD
    Institution VARCHAR(100),     -- Name of the institution
    Years INTEGER,                -- Year the degree was completed
    Speciality VARCHAR(100)       -- Field of study
);

/* Sample rows:
ID | ID_Usr | educational_level | Institution | Years | Speciality
1  | 1      | 2                 | MIT         | 2016  | Computer Science
2  | 2      | 1                 | UCLA        | 2019  | Business
3  | 3      | 3                 | Stanford    | 2014  | Data Science
*/
"""}]

In [4]:
context.append({'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient, using valid SQLite.
-- Here are some example questions and their correct SQL queries:

-- Q: Who is the best paid employee?
-- A:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
ORDER BY s.salary DESC
LIMIT 1;

-- Q: What is the average salary per department?
-- A:
SELECT e.department, AVG(s.salary) AS avg_salary
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
GROUP BY e.department;

-- Q: Which institution has the highest average salary among its graduates?
-- A:
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_Usr = sa.ID_Usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
"""
})

In [5]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [6]:
# new prompt 
context_user = context.copy()
print(return_CCRMSQL("Who is the best paid employee?", context_user))

SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
ORDER BY s.salary DESC
LIMIT 1;


In [7]:
# old prompt 
old_context_user = old_context.copy()
print(return_CCRMSQL("Who is the best paid employee?", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name and salary of the employee who is the best paid by joining the "employees" and "salary" tables on the ID_usr column, ordering the results by salary in descending order, and limiting the output to only the top result.


In [8]:
#new
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", context_user))


SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN salary sa ON st.ID_Usr = sa.ID_Usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;


In [9]:
#old
print(return_CCRMSQL("Which institution has the highest average salary among its graduates?", old_context_user))


This is your SQL:
```sql
SELECT s.Institution, AVG(sa.salary) AS avg_salary
FROM studies s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the tables `studies`, `employees`, and `salary` on the user ID to calculate the average salary for each institution. It then selects the institution with the highest average salary among its graduates.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [ ]:
#Employees hired before 2019 with a PhD
print(return_CCRMSQL(
    "List the names of employees hired before 2019 who have a PhD.",
    context_user
))

SELECT e.name
FROM employees e
JOIN studies s ON e.ID_Usr = s.ID_Usr
WHERE e.hire_date < '2019-01-01' AND s.educational_level = 3;


In [11]:
# Total compensation (salary + bonus) per employee in 2023
print(return_CCRMSQL(
    "What is the total compensation (salary + bonus) for each employee in 2023, ordered from highest to lowest?",
    context_user
))

SELECT e.name, (s.salary + s.bonus) AS total_compensation
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.year = 2023
ORDER BY total_compensation DESC;


In [12]:
# Out-of-schema question (testa alucinação)
print(return_CCRMSQL(
    "How many vacation days does each employee have?",
    context_user
))

SELECT e.name, (2024 - strftime('%Y', e.hire_date)) * 14 AS vacation_days
FROM employees e;


The new prompt style using CREATE TABLE syntax with sample rows and few-shot examples produced significantly cleaner and more accurate SQL compared to the old JSON-based prompt — especially for multi-table JOINs. Few-shot samples act as implicit instructions, guiding the model toward the expected query structure without needing verbose explanations. The main weakness observed was with out-of-schema questions, where the model occasionally hallucinated column names instead of refusing to answer.
